# Dashboard — Dev Log

## Objetivo e papel no pipeline

Este modulo (`apps/dashboard/`) e a **camada de apresentacao** do AthenaGov AI: um app
Streamlit que da visibilidade humana a tudo que os outros modulos do V1 fazem — sem
reimplementar nenhuma logica de negocio propria. Ele cobre dois papeis distintos:

1. **Observabilidade do proprio projeto** (secao "Visao geral"): le `status/*.json`
   direto do disco e mostra o progresso agregado das 10 capacidades do V1 do
   `ROADMAP.md` — quantas estao `done`, quantos testes verdes no total, etc. Isso
   funciona **mesmo sem nenhum backend rodando**.
2. **Interface operacional para o Governance Copilot** (as outras 4 secoes): formularios
   e telas que chamam a API HTTP do backend (`core/governance_copilot`) para gerar um
   RIPD, escanear PII, escanear seguranca de prompt e consultar a trilha de auditoria.

Toda a comunicacao com o backend passa exclusivamente por
`apps/dashboard/client.py::GovernanceCopilotClient`, um cliente HTTP fino sobre
`httpx`, que fala o contrato de API abaixo e devolve os tipos Pydantic reais de
`shared/schemas.py` — nunca dicts soltos, nunca tipos redefinidos localmente.

**Contrato de API consumido** (implementado por `core/governance_copilot`):

```text
GET  /health
POST /api/v1/pii/detect            {"text": str} -> PIIDetectionResult
POST /api/v1/prompt-security/scan  {"prompt": str} -> PromptSecurityResult
POST /api/v1/policy/evaluate       {"data_categories": list[str], "legal_basis": str, "context": dict|null} -> list[PolicyDecision]
POST /api/v1/ripd/generate         {"project_name": str, "project_description": str, "data_categories": list[str], "legal_basis": str, "context": dict|null} -> RIPDReport
GET  /api/v1/audit/verify -> {"valid": bool}
GET  /api/v1/audit/events?limit=50 -> list[AuditEvent]
```

## Decisoes de design

**Por que Streamlit, e nao um frontend JS separado (React/Vue)?**

- O publico deste dashboard e interno/compliance (times de privacidade, DPO,
  engenharia), nao usuarios finais de um produto — um app de dados/formularios em
  Python, sem build step de frontend, sem estado de bundler, e o suficiente e muito
  mais rapido de entregar no escopo do V1.
- Streamlit da tabelas, formularios, metricas e JSON viewer prontos (`st.table`,
  `st.form`, `st.metric`, `st.json`) exatamente para o formato de dado que a API
  devolve (Pydantic -> JSON) — sem precisar escrever nenhum componente de UI do zero.
- Todo o projeto ja e Python (core/*, shared/*); manter o dashboard em Python evita
  duplicar os `enums` e `schemas` de `shared/schemas.py` numa segunda linguagem/runtime.

**Por que desacoplado via contrato HTTP, e nunca por import direto de
`core/governance_copilot`?**

- **Restricao real deste ciclo**: o backend (`core/governance_copilot`, API FastAPI)
  estava sendo construido em paralelo por outro agente e **nao existia no disco**
  durante o desenvolvimento deste modulo. Um import direto (`from
  core.governance_copilot import ...`) teria quebrado a qualquer momento e tornado
  o dashboard impossivel de testar de forma isolada.
- **Isso permitiu construir em paralelo**: com o contrato de API fixado antecipadamente
  (rotas, metodos, shapes de request/response — todos espelhados em
  `shared/schemas.py`), este modulo foi implementado e testado **inteiramente contra
  um `httpx.MockTransport`**, sem nenhuma dependencia de tempo de execucao do backend
  real. Quando o backend subir, so a URL (`ATHENAGOV_API_URL`) precisa apontar para ele
  — nenhuma linha de codigo do dashboard muda.
- **Fronteira de processo tambem faz sentido em producao**: o dashboard e um cliente
  HTTP como qualquer outro consumidor da API (CLI, integracao externa, etc.), o que
  evita acoplamento acidental ao `sys.path`/versao instalada do backend e permite
  deploy independente (o dashboard pode escalar, reiniciar ou fazer rollback sem
  afetar o backend, e vice-versa).
- **Trade-off aceito**: nao ha type-checking estatico entre o schema que o backend
  realmente devolve e o que o dashboard espera — so validacao em runtime via
  `pydantic.model_validate` (que falha ruidosamente se o contrato divergir, o que e
  aceitavel dado que ambos os lados importam os MESMOS tipos de `shared/schemas.py`).

**Por que extrair a logica em funcoes puras (`apps/dashboard/app.py`) separadas do
codigo `streamlit.*`?**

- Streamlit nao roda de forma significativa fora de `streamlit run` (nao ha um jeito
  simples de exercitar `st.button`/`st.form` num teste pytest comum). Em vez de deixar
  a logica de negocio (parsing de `status/*.json`, agregacao de progresso, formatacao
  de achados de PII/policy/audit em linhas de tabela) misturada com chamadas `st.*`,
  ela foi extraida em funcoes puras (`compute_v1_progress`, `pii_findings_to_rows`,
  `ripd_report_summary`, etc.) — testadas de verdade, com as funcoes `render_*` como
  uma casca fina que so chama essas funcoes e desenha o resultado.

## Setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from apps.dashboard.client import (
    GovernanceCopilotClient,
    GovernanceCopilotConnectionError,
    GovernanceCopilotHTTPError,
)
from apps.dashboard.app import (
    compute_v1_progress,
    load_all_statuses,
    build_overview_rows,
    ripd_report_summary,
    describe_client_error,
)

print("Modulo carregado:", GovernanceCopilotClient.__module__)

Modulo carregado: apps.dashboard.client


## Demonstracao do `GovernanceCopilotClient` contra um mock transport

Nenhuma chamada de rede real acontece abaixo — um `httpx.MockTransport` simula as
respostas do backend `core/governance_copilot` exatamente no formato definido em
`shared/schemas.py` (o mesmo contrato que os testes em `apps/dashboard/tests/`
usam). Isso reproduz, de forma reprodutivel, como o dashboard se comporta quando
conversa com a API real.

In [2]:
import json
import httpx

PII_PAYLOAD = {
    "findings": [
        {
            "entity_type": "CPF",
            "text_span": "111.444.777-35",
            "start": 10,
            "end": 25,
            "category": "personal",
            "confidence": 0.98,
        }
    ],
    "has_sensitive_data": False,
    "summary": "1 achado(s) - CPF=1",
}

RIPD_PAYLOAD = {
    "project_name": "Chatbot de Atendimento",
    "generated_at": "2026-08-19T12:00:00",
    "data_categories": ["personal"],
    "legal_basis": "consent",
    "pii_result": PII_PAYLOAD,
    "policy_decisions": [
        {
            "policy_id": "POL-1",
            "status": "allow_with_mitigation",
            "rationale": "Dado pessoal com base legal valida, requer anonimizacao.",
            "mitigations": ["Anonimizar antes de armazenar"],
            "risk_level": "medium",
        }
    ],
    "prompt_security": None,
    "trust_score": {
        "score": 82.5,
        "risk_level": "low",
        "components": {"pii_exposure": 0.9, "policy_compliance": 0.8},
        "explanation": None,
    },
    "regulatory_context": [
        {"source": "LGPD", "article": "Art. 7\u00ba", "text": "Bases legais...", "score": 0.91}
    ],
    "mitigations": ["Anonimizar antes de armazenar"],
    "executive_summary": "Projeto de baixo risco, com uma mitigacao recomendada.",
}


def mock_handler(request: httpx.Request) -> httpx.Response:
    path = request.url.path
    print(f">> {request.method} {path}")
    if request.content:
        print("   request body:", json.loads(request.content))
    if path == "/api/v1/pii/detect":
        return httpx.Response(200, json=PII_PAYLOAD)
    if path == "/api/v1/ripd/generate":
        return httpx.Response(200, json=RIPD_PAYLOAD)
    if path == "/api/v1/audit/verify":
        return httpx.Response(200, json={"valid": True})
    return httpx.Response(404, json={"detail": "rota nao simulada neste demo"})


client = GovernanceCopilotClient(
    base_url="http://localhost:8000", transport=httpx.MockTransport(mock_handler)
)
print("Client criado, base_url =", client.base_url)

Client criado, base_url = http://localhost:8000


In [3]:
# Exemplo real 1: deteccao de PII
resultado_pii = client.detect_pii(
    "Cliente Joao da Silva, CPF 111.444.777-35, telefone (11) 91234-5678."
)
print()
print("Resposta parseada (PIIDetectionResult):")
print(resultado_pii.model_dump_json(indent=2))

>> POST /api/v1/pii/detect
   request body: {'text': 'Cliente Joao da Silva, CPF 111.444.777-35, telefone (11) 91234-5678.'}

Resposta parseada (PIIDetectionResult):
{
  "findings": [
    {
      "entity_type": "CPF",
      "text_span": "111.444.777-35",
      "start": 10,
      "end": 25,
      "category": "personal",
      "confidence": 0.98
    }
  ],
  "has_sensitive_data": false,
  "summary": "1 achado(s) - CPF=1"
}


In [4]:
# Exemplo real 2: geracao de RIPD
ripd = client.generate_ripd(
    project_name="Chatbot de Atendimento",
    project_description="Chatbot que responde duvidas de clientes usando dados de cadastro.",
    data_categories=["personal"],
    legal_basis="consent",
)
print()
print("Resumo formatado para a UI (ripd_report_summary):")
print(json.dumps(ripd_report_summary(ripd), indent=2, ensure_ascii=False))

>> POST /api/v1/ripd/generate
   request body: {'project_name': 'Chatbot de Atendimento', 'project_description': 'Chatbot que responde duvidas de clientes usando dados de cadastro.', 'data_categories': ['personal'], 'legal_basis': 'consent', 'context': None}

Resumo formatado para a UI (ripd_report_summary):
{
  "project_name": "Chatbot de Atendimento",
  "generated_at": "2026-08-19T12:00:00",
  "legal_basis": "consent",
  "data_categories": [
    "personal"
  ],
  "trust_score": 82.5,
  "trust_risk_level": "low",
  "pii_findings_count": 1,
  "has_sensitive_data": false,
  "policy_decisions_count": 1,
  "denied_policies_count": 0,
  "mitigations_count": 1,
  "regulatory_context_count": 1,
  "executive_summary": "Projeto de baixo risco, com uma mitigacao recomendada."
}


In [5]:
# Exemplo real 3: tratamento de erro quando a API esta fora do ar
def connection_refused_handler(request: httpx.Request) -> httpx.Response:
    raise httpx.ConnectError("Connection refused", request=request)

client_offline = GovernanceCopilotClient(
    base_url="http://localhost:8000",
    transport=httpx.MockTransport(connection_refused_handler),
)

try:
    client_offline.verify_audit()
except GovernanceCopilotConnectionError as exc:
    print("Excecao capturada:", type(exc).__name__)
    print("Mensagem exibida na UI:", describe_client_error(exc))

Excecao capturada: GovernanceCopilotConnectionError
Mensagem exibida na UI: API do Governance Copilot indisponível. Detalhe: Não foi possível conectar à API do Governance Copilot em http://localhost:8000. Verifique se o backend está rodando.


## Demonstracao da secao "Visao geral" (le `status/*.json` real do disco)

Diferente das secoes acima, esta parte do dashboard NAO depende da API — ela le os
arquivos `status/*.json` reais deste repositorio, no disco, agora.

In [6]:
statuses = load_all_statuses(REPO_ROOT / "status")
progress = compute_v1_progress(statuses)
print("Progresso agregado do V1:")
print(json.dumps(progress, indent=2, ensure_ascii=False))
print()
for row in build_overview_rows(statuses):
    print(row)

Progresso agregado do V1:
{
  "total_modules": 10,
  "counts": {
    "planned": 2,
    "in_progress": 0,
    "done": 8,
    "blocked": 0
  },
  "percent_done": 80.0,
  "tests_passed_total": 190,
  "tests_total_total": 190
}

{'Capacidade': 'Policy Engine', 'Pasta': 'core/policy_engine/', 'Status': '✅ concluído', 'Testes': '25/25', 'Atualizado em': '2026-08-19T13:23:22'}
{'Capacidade': 'PII Detection', 'Pasta': 'core/pii_detection/', 'Status': '✅ concluído', 'Testes': '25/25', 'Atualizado em': '2026-08-19T00:45:00'}
{'Capacidade': 'Prompt Security (injection/jailbreak)', 'Pasta': 'core/prompt_security/', 'Status': '✅ concluído', 'Testes': '39/39', 'Atualizado em': '2026-08-19T00:00:00'}
{'Capacidade': 'Explainability', 'Pasta': 'core/explainability/', 'Status': '✅ concluído', 'Testes': '12/12', 'Atualizado em': '2026-08-19T00:15:00'}
{'Capacidade': 'AI Trust Score', 'Pasta': 'core/trust_score/', 'Status': '✅ concluído', 'Testes': '20/20', 'Atualizado em': '2026-08-19T13:29:13'}
{'Capaci

## Suite de testes

Executando a suite pytest real via `subprocess`, a partir da raiz do repo, com o
Python do venv do projeto (`C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe`).
Todos os testes cobrem `GovernanceCopilotClient` (via `httpx.MockTransport`, sem rede
real) e as funcoes puras de `apps/dashboard/app.py`.

In [7]:
import subprocess

python_exe = r"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe"
proc = subprocess.run(
    [python_exe, "-m", "pytest", "apps/dashboard/tests", "-v"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)
print(proc.stdout[-4000:])
if proc.returncode != 0:
    print("STDERR:", proc.stderr[-2000:])
print("Return code:", proc.returncode)

ompute_v1_progress_handles_empty_statuses_without_division_error PASSED [ 18%]
apps/dashboard/tests/test_app.py::test_status_badge[done-conclu\xeddo] PASSED [ 20%]
apps/dashboard/tests/test_app.py::test_status_badge[in_progress-andamento] PASSED [ 22%]
apps/dashboard/tests/test_app.py::test_status_badge[blocked-bloqueado] PASSED [ 24%]
apps/dashboard/tests/test_app.py::test_status_badge[planned-planejado] PASSED [ 26%]
apps/dashboard/tests/test_app.py::test_status_badge[None-planejado] PASSED [ 28%]
apps/dashboard/tests/test_app.py::test_status_badge[unrecognized-planejado] PASSED [ 30%]
apps/dashboard/tests/test_app.py::test_build_overview_rows_with_and_without_data PASSED [ 32%]
apps/dashboard/tests/test_app.py::test_describe_client_error_for_connection_error PASSED [ 34%]
apps/dashboard/tests/test_app.py::test_describe_client_error_for_http_error PASSED [ 36%]
apps/dashboard/tests/test_app.py::test_describe_client_error_for_generic_exception PASSED [ 38%]
apps/dashboard/tests/test_a

## Handoff Summary

**Contrato de API consumido** (implementado por `core/governance_copilot`, base URL
configuravel via `ATHENAGOV_API_URL`, default `http://localhost:8000`):

```text
GET  /health -> {"status": "ok"}
POST /api/v1/pii/detect            {"text": str} -> PIIDetectionResult
POST /api/v1/prompt-security/scan  {"prompt": str} -> PromptSecurityResult
POST /api/v1/policy/evaluate       {"data_categories": list[str], "legal_basis": str, "context": dict|null} -> list[PolicyDecision]
POST /api/v1/ripd/generate         {"project_name": str, "project_description": str, "data_categories": list[str], "legal_basis": str, "context": dict|null} -> RIPDReport
GET  /api/v1/audit/verify -> {"valid": bool}
GET  /api/v1/audit/events?limit=50 -> list[AuditEvent]
```

**Como rodar:**

```bash
# a partir da raiz do repo, com o venv do projeto ativo
streamlit run apps/dashboard/app.py

# apontar para uma API em outro host/porta (PowerShell):
$env:ATHENAGOV_API_URL = "http://localhost:9000"; streamlit run apps/dashboard/app.py
# ou via campo "URL da API" na sidebar do app, em tempo de execucao
```

**Como rodar os testes:**

```bash
C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe -m pytest apps/dashboard/tests -v
```

**Assinaturas publicas exatas:**

```python
class GovernanceCopilotClient:
    def __init__(self, base_url: str | None = None, timeout: float = 15.0,
                 transport: httpx.BaseTransport | None = None) -> None: ...
    def health(self) -> dict: ...
    def detect_pii(self, text: str) -> PIIDetectionResult: ...
    def scan_prompt_security(self, prompt: str) -> PromptSecurityResult: ...
    def evaluate_policy(self, data_categories: list[str], legal_basis: str,
                         context: dict | None = None) -> list[PolicyDecision]: ...
    def generate_ripd(self, project_name: str, project_description: str,
                       data_categories: list[str], legal_basis: str,
                       context: dict | None = None) -> RIPDReport: ...
    def verify_audit(self) -> dict: ...
    def get_audit_events(self, limit: int = 50) -> list[AuditEvent]: ...
```

**Limitacoes (V1):**

- As 4 secoes que dependem da API (Gerador de RIPD, Scanner de PII, Scanner de
  Prompt Security, Auditoria) **nao foram validadas ponta-a-ponta contra um backend
  real** neste ciclo, porque `core/governance_copilot` ainda nao existia no disco
  durante o desenvolvimento deste modulo. Foram validadas via `httpx.MockTransport`
  (testes automatizados + demo reproduzida acima). Quando o backend subir, validar
  manualmente pelo menos um fluxo completo de cada tela.
- Streamlit nao e exercitado por pytest — as funcoes `render_*`/`main` de `app.py`
  nao tem cobertura de teste direta (limitacao da ferramenta, nao do codigo); toda
  logica nao-trivial que elas usam foi extraida em funcoes puras testadas.
- Sem autenticacao/autorizacao na chamada a API (nenhum header de auth enviado) —
  adequado ao escopo V1 (uso interno, API sem auth definida ainda no contrato).
- Sem paginacao real na tela de Auditoria — usa o parametro `limit` simples do
  contrato (`GET /api/v1/audit/events?limit=N`), sem cursor/offset.
- Sem cache/estado entre chamadas (cada acao gera uma nova requisicao HTTP) — para
  o volume de uso esperado do V1 (uso manual, interno), isso e aceitavel.

**TODO V2 (ver ROADMAP.md):**

- Validacao ponta-a-ponta com o backend real assim que `core/governance_copilot`
  estiver disponivel (TODO explicito, nao feito neste ciclo).
- Autenticacao na chamada a API, quando o contrato do backend definir isso.
- Visualizacoes mais ricas para `TrustScoreResult`/`ExplainabilityResult` (ex.
  grafico de `components`/`factors` em vez de so numeros/JSON).
- Paginacao real na tela de Auditoria para historicos grandes.
- Cache leve (`st.cache_data`) para reduzir chamadas repetidas na secao de
  Auditoria/Visao geral, se o volume de uso justificar.